# PneumoVision: text training (RR + DN), Task B phenotyping

**Before starting:** Runtime → Change runtime type → **T4 GPU**.

Run the cells **top to bottom**. If Colab disconnects: reconnect and run all cells again.
Training continues from the last finished epoch (it does not start over).

Patient data rule: this notebook prints only counts, shapes and scores, never note text or patient rows.

## 1. Settings (the only cell you may need to edit)

In [ ]:
# Where Caroll's shared folder appears after you add the shortcut to My Drive
SHARED = '/content/drive/MyDrive/PneumoVision_shared/data'
# Folder that holds the code zip (a shortcut to another account's folder is fine: it is only read)
MINE = '/content/drive/MyDrive/PneumoVision_farida'
# Where checkpoints, logs and the BioBERT cache are SAVED. A shortcut to another account's folder works:
# new files are OWNED by the account you mounted and use its storage, wherever the folder lives.
OUT = MINE

SEED = 1002          # keep 1002 for the first run (the repo's default)
LR = 0.001           # the repo scripts' value
# Text model. Paper says BioBERT. (Repo code default was emilyalsentzer/Bio_ClinicalBERT.)
BERT = 'dmis-lab/biobert-v1.1'

## 2. Check the GPU and connect Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 3. Check every needed file is on Drive

In [ ]:
import os, glob
need = {
    'code zip':        f'{MINE}/medpatch_code.zip',
    'all_stays':       f'{SHARED}/ehr/root/all_stays.csv',
    'train listfile':  f'{SHARED}/ehr/phenotyping/train_listfile.csv',
    'val listfile':    f'{SHARED}/ehr/phenotyping/val_listfile.csv',
    'test listfile':   f'{SHARED}/ehr/phenotyping/test_listfile.csv',
    'timeseries rar':  f'{SHARED}/ehr/phenotyping/phenotype_labels.rar',
}
def first_existing(*names):
    hits = [f'{SHARED}/notes/{n}' for n in names if os.path.exists(f'{SHARED}/notes/{n}')]
    return hits[0] if hits else None
RAD = first_existing('radiology.zip', 'radiology.csv', 'radiology.csv.gz')
DIS = first_existing('discharge.zip', 'discharge.csv', 'discharge.csv.gz')
ok = True
for name, p in need.items():
    exists = os.path.exists(p)
    ok &= exists
    print(f"OK       {name:15s} {p}  {os.path.getsize(p)/1e6:,.0f} MB" if exists else f"MISSING  {name:15s} {p}")
print(f"{'OK     ' if RAD else 'MISSING'}  radiology       {RAD}")
print(f"{'OK     ' if DIS else 'MISSING'}  discharge       {DIS}")
ok &= RAD is not None
assert ok, 'Fix the MISSING files above before continuing (RR needs everything except discharge).'

## 4. Install packages and unpack the code (about 2 minutes)

In [ ]:
# 4.45.2, not the repo's 4.44.2: Colab is on Python 3.13, which has no prebuilt tokenizers 0.19 wheel
!pip -q install transformers==4.45.2 timm==1.0.9 einops wandb
!rm -rf /content/medpatch && mkdir -p /content/medpatch
!unzip -q -o "{MINE}/medpatch_code.zip" -d /content/medpatch
!md5sum "{MINE}/medpatch_code.zip" | tee /content/code_md5.txt
import os
os.environ['WANDB_MODE'] = 'offline'                 # no wandb login needed
os.environ['PYTHONUNBUFFERED'] = '1'                 # show progress lines immediately
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'  # repo checkpoints are saved with numpy values
os.environ['TOKENIZERS_PARALLELISM'] = 'false'       # silences the "process just got forked" warning
import torch; print('torch', torch.__version__, 'GPU:', torch.cuda.is_available())

## 5. Prepare the data on Colab's local disk (about 5–15 minutes)
Unpacks the timeseries, removes the extra MIMIC-IV v3 patients (expected: 42,328 / 4,756 / 11,845 rows),
and copies the notes locally (much faster than reading from Drive). Drive files are not changed.

In [ ]:
import os, glob
!apt-get -qq install -y unrar > /dev/null
if not os.path.exists('/content/ts_done'):
    !mkdir -p /content/ts && unrar x -o+ -inul "{SHARED}/ehr/phenotyping/phenotype_labels.rar" /content/ts/ && touch /content/ts_done
# find the folder that holds train/ and test/ (works whatever the rar's inner folder name is)
hits = glob.glob('/content/ts/**/train/14991576_episode3_timeseries.csv', recursive=True)
assert hits, 'Could not find the train/ timeseries folder inside the rar'
TS = os.path.dirname(os.path.dirname(hits[0]))
print('timeseries folder:', TS, '| train files:', len(os.listdir(f'{TS}/train')), '| test files:', len(os.listdir(f'{TS}/test')))

common = (f'--all_stays "{SHARED}/ehr/root/all_stays.csv" --listfile_dir "{SHARED}/ehr/phenotyping" '
          f'--timeseries_dir "{TS}" --repo /content/medpatch')
!rm -rf /content/ehr /content/ehr_small
!python /content/medpatch/prepare_data.py {common} --out /content/ehr
!python /content/medpatch/prepare_data.py {common} --out /content/ehr_small --small 64 > /dev/null

!mkdir -p /content/notes

def unpack_note(src, name):
    # puts the file at /content/notes/<name>.csv whatever the upload format (.zip / .csv.gz / .csv)
    dst = f'/content/notes/{name}.csv'
    if src is None or os.path.exists(dst):
        return
    if src.endswith('.zip'):
        !rm -rf /content/tmp_unzip && mkdir -p /content/tmp_unzip && unzip -q -o "{src}" -d /content/tmp_unzip
        csvs = [p for p in glob.glob('/content/tmp_unzip/**/*.csv', recursive=True) if '__MACOSX' not in p]
        assert len(csvs) == 1, f'expected exactly one csv inside {src}, found {len(csvs)}'
        os.rename(csvs[0], dst)
    elif src.endswith('.gz'):
        !gunzip -c "{src}" > "{dst}"
    else:
        !cp "{src}" "{dst}"

unpack_note(RAD, 'radiology')
unpack_note(DIS, 'discharge')
# must be the ORIGINAL PhysioNet files (column names only are printed)
import pandas as pd
for n in ['radiology', 'discharge']:
    f = f'/content/notes/{n}.csv'
    if os.path.exists(f):
        cols = list(pd.read_csv(f, nrows=0).columns)
        good = {'subject_id', 'hadm_id', 'charttime', 'text'} <= set(cols)
        print(f'{n}.csv columns: {cols} -> {"OK" if good else "WRONG FILE (need the original with a text column)"}')
!ls -la /content/notes

## 6. Training helper (run this cell, nothing happens yet)

In [ ]:
import datetime, subprocess, os

def train(modality, ehr_dir='/content/ehr', run_name=None, epochs=100, resume=True):
    mod = modality.upper()            # 'RR' or 'DN'
    m = mod.lower()
    run_name = run_name or f'taskB_unimodal_{m}_biobert_seed{SEED}'
    run_dir = f'{OUT}/runs/{run_name}'
    os.makedirs(run_dir, exist_ok=True)
    cmd = (f'python fusion_main.py --mode train --epochs {epochs} --batch_size 16 --lr {LR} '
           f'--num_classes 25 --modalities EHR-{mod} --fusion_type unimodal_{m} --text_encoder biobert '
           f'--classifier mlp --loss bce --task phenotyping --labels_set pheno --output_dim_{m} 512 '
           f'--data_pairs paired --seed {SEED} --num_workers 2 --bert_model_name {BERT} '
           f'--ehr_data_dir {ehr_dir} --notes_data_dir /content/notes --cxr_data_dir NONE '
           f'--save_dir "{run_dir}/checkpoints"' + (' --resume' if resume else ''))
    with open(f'{run_dir}/command.txt', 'a') as f:
        f.write(f'# {datetime.datetime.now():%Y-%m-%d %H:%M}  code md5: {open("/content/code_md5.txt").read().split()[0]}\n{cmd}\n\n')
    print('RUN FOLDER:', run_dir)
    print(cmd)
    # output goes to the screen AND to train.log on Drive (appended, so resumed runs keep history)
    !cd /content/medpatch && {cmd} 2>&1 | grep --line-buffered -v "Warning\|warn(" | tee -a "{run_dir}/train.log"
    print('\nResults files:');
    # the repo does not write a results file; the test summary is in train.log
    lines = open(f'{run_dir}/train.log').readlines()
    tests = [i for i, l in enumerate(lines) if 'test' in l and 'best mean auc' in l]
    if tests:
        print('\nFINAL TEST RESULT (from train.log):')
        print(''.join(lines[tests[-1] - 28: tests[-1] + 8]))  # 25 disease lines + 3 blank lines above

## 7. FAST path: write the two scripts and the helper (run this, nothing trains yet)
BioBERT is frozen, so its output for a patient never changes. The fast path runs BioBERT **once** per patient,
saves one vector per patient to Drive (`PneumoVision_farida/cache/`), then trains only the small layer on top
(same model, optimizer, scheduler, early stopping as the repo), with the paper's 10-learning-rate search.
Difference vs the repo: BioBERT dropout is off while training (the repo leaves it on). Write that in the Sheet.

In [ ]:
!mkdir -p /content/fast_text

In [ ]:
%%writefile /content/fast_text/cache_embeddings.py
"""Fast path, step 1: run the frozen text encoder ONCE and cache one 768-d vector per stay.

Why this is equivalent to the repo (models/fusion.py UnimodalRR / UnimodalDN + models/text_models.py):
  repo:  chunks(512, stride 0, padding=max_length) -> BERT -> mean over chunks -> fc (Linear 768->512)
         -> mean over the 512 token positions -> MLPClassifier
  here:  chunks -> BERT -> mean over chunks -> mean over the 512 positions  (cached)
         and the head applies fc -> MLPClassifier.
  fc is affine, so mean_t(fc(x_t)) == fc(mean_t(x_t)) exactly (up to float rounding).
  Texts and labels come from the repo's own loadmetadata() + listfile merge (DataFusion.py), unchanged.
Known difference: the repo leaves BERT dropout ON during training epochs (model.train()); the cache is computed
with BERT in eval mode, which is exactly what the repo uses for validation/test.

Output (derived patient data -> must stay under data/, which is gitignored):
  <out>/<split>_X.npy (N x 768 float32), <split>_y.npy (N x 25), <split>_stay_id.npy, meta.json
Prints only counts and timings.
"""
import argparse
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import torch

ap = argparse.ArgumentParser()
ap.add_argument('--repo', required=True, help='path to medpatch/')
ap.add_argument('--modality', choices=['RR', 'DN'], required=True)
ap.add_argument('--bert_model_name', default='dmis-lab/biobert-v1.1')
ap.add_argument('--ehr_data_dir', required=True)
ap.add_argument('--notes_data_dir', required=True)
ap.add_argument('--notes_file', default=None, help='override the notes csv name (e.g. a cleaned/cut variant)')
ap.add_argument('--out', required=True)
ap.add_argument('--chunk_batch', type=int, default=32, help='BERT chunks per forward pass (lower if GPU OOM)')
ap.add_argument('--limit', type=int, default=0, help='only the first N stays per split (timing test)')
ap.add_argument('--splits', nargs='+', default=['train', 'val', 'test'], help='which splits to (re)compute')
cli = ap.parse_args()
PART = 2048  # stays per saved part file (resume unit after a disconnect)

sys.path.insert(0, os.path.abspath(cli.repo))
from datasets.DataFusion import loadmetadata, CLASSES  # noqa: E402
from transformers import BertModel, BertTokenizerFast  # noqa: E402

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '')
print('TEXT ENCODER MODEL:', cli.bert_model_name)

# --- same data path as fusion_main.py with --modalities EHR-<M> --data_pairs paired --task phenotyping
args = argparse.Namespace(modalities=f'EHR-{cli.modality}', data_pairs='paired', task='phenotyping',
                          cxr_data_dir='NONE', ehr_data_dir=cli.ehr_data_dir, labels_set='pheno')
cols = ['subject_id', 'hadm_id', 'charttime', 'text']
empty = pd.DataFrame(columns=cols)
fname = cli.notes_file or ('discharge.csv' if cli.modality == 'DN' else 'radiology.csv')
notes = pd.read_csv(os.path.join(cli.notes_data_dir, fname), usecols=cols)
print(f'notes file {fname}: {len(notes)} rows')
dn, rr = (notes, empty) if cli.modality == 'DN' else (empty, notes)
meta = loadmetadata(args, dn, rr, None)
text_col = 'discharge_text' if cli.modality == 'DN' else 'radiology_text'

tok = BertTokenizerFast.from_pretrained(cli.bert_model_name)
bert = BertModel.from_pretrained(cli.bert_model_name).to(device).eval()
hidden = bert.config.hidden_size


@torch.no_grad()
def encode(texts):
    """Same tokenisation as Text_encoder.process_notes; returns per-note mean over chunks and 512 positions."""
    enc = tok(texts, padding='max_length', truncation=True, max_length=512, stride=0,
              return_overflowing_tokens=True, return_tensors='pt')
    mapping = enc['overflow_to_sample_mapping'].numpy()
    chunk_vecs = []
    for s in range(0, len(mapping), cli.chunk_batch):
        ids = enc['input_ids'][s:s + cli.chunk_batch].to(device)
        mask = enc['attention_mask'][s:s + cli.chunk_batch].to(device)
        h = bert(input_ids=ids, attention_mask=mask).last_hidden_state  # [c, 512, 768]
        chunk_vecs.append(h.mean(dim=1).float().cpu())                    # mean over all 512 positions
    chunk_vecs = torch.cat(chunk_vecs).numpy()
    out = np.zeros((len(texts), hidden), dtype=np.float32)
    for i in range(len(texts)):
        out[i] = chunk_vecs[mapping == i].mean(axis=0)                    # mean over that note's chunks
    return out, len(mapping)


os.makedirs(cli.out, exist_ok=True)
info = {'model': cli.bert_model_name, 'modality': cli.modality, 'notes_file': fname, 'splits': {}}
if os.path.exists(f'{cli.out}/meta.json'):                      # keep info about splits done earlier
    old = json.load(open(f'{cli.out}/meta.json'))
    assert old['model'] == cli.bert_model_name and old['notes_file'] == fname, 'cache folder holds a different model/notes'
    info['splits'].update(old.get('splits', {}))
for split in cli.splits:
    lf = pd.read_csv(f'{cli.ehr_data_dir}/phenotyping/{split}_listfile.csv')
    m = meta.merge(lf, how='inner', on='stay_id')      # identical to load_cxr_ehr_rr_dn
    if cli.limit:
        m = m.head(cli.limit)
    texts = ['' if pd.isna(t) else str(t) for t in m[text_col].values]  # identical to my_collate
    n_chunks, t0, new = 0, time.time(), 0
    step = 64
    # work in parts of PART stays; a finished part is saved, so a rerun after a disconnect skips it
    parts = [f'{cli.out}/{split}_part{p // PART:03d}_of{len(texts)}.npy' for p in range(0, len(texts), PART)]
    for k, part in enumerate(parts):
        if os.path.exists(part):
            continue
        lo, hi = k * PART, min((k + 1) * PART, len(texts))
        Xp = np.zeros((hi - lo, hidden), dtype=np.float32)
        for s in range(lo, hi, step):
            Xp[s - lo:s - lo + step], c = encode(texts[s:min(s + step, hi)])
            n_chunks += c
            new += len(texts[s:min(s + step, hi)])
            if (s // step) % 50 == 0:
                el = time.time() - t0
                print(f'  {split}: {min(s + step, hi)}/{len(texts)} stays, {n_chunks} chunks, {el / 60:.1f} min, '
                      f'est. left {el / new * (len(texts) - min(s + step, hi)) / 60:.1f} min', flush=True)
        np.save(part + '.tmp.npy', Xp)
        os.replace(part + '.tmp.npy', part)  # the part file only appears once it is complete
    X = np.concatenate([np.load(p) for p in parts]) if parts else np.zeros((0, hidden), dtype=np.float32)
    assert len(X) == len(texts)
    np.save(f'{cli.out}/{split}_X.npy', X)
    for p in parts:
        os.remove(p)
    np.save(f'{cli.out}/{split}_y.npy', m[CLASSES].values.astype(np.float32))
    np.save(f'{cli.out}/{split}_stay_id.npy', m['stay_id'].values)
    info['splits'][split] = {'stays': len(texts), 'chunks': int(n_chunks), 'minutes': round((time.time() - t0) / 60, 2)}
    print(f'{split}: {len(texts)} stays ({new} encoded this run), {n_chunks} chunks, {(time.time() - t0) / 60:.1f} min')
    json.dump(info, open(f'{cli.out}/meta.json', 'w'), indent=1)  # saved after every split
json.dump(info, open(f'{cli.out}/meta.json', 'w'), indent=1)
print('cache written to', os.path.abspath(cli.out))

In [ ]:
%%writefile /content/fast_text/train_head.py
"""Fast path, step 2: train the RR/DN head on cached embeddings, with the paper's LR search.

Mirrors the repo's unimodal_rr / unimodal_dn training (trainers/MSMA_trainer.py):
  model      fc = Linear(768 -> 512)  then  Classifier('mlp') = LayerNorm(512) + Linear(512 -> 25)
  loss       BCEWithLogits;  Adam(lr, betas=(0.9, 0.9))  [repo passes beta_1=0.9 as beta2]
  scheduler  ReduceLROnPlateau(factor=0.5, patience=10, mode='min') stepped on validation loss
  loop       up to 100 epochs; validate BEFORE each train epoch; keep best val mean AUROC;
             early stop after 15 epochs without improvement; batch 16, shuffle, drop_last
Paper (§4.4): 10 sweeps, LR sampled in [1e-5, 1e-3], select on validation AUROC.
Unspecified in the paper: the sampling distribution -> log-uniform here (stated in the output).
The test set is evaluated ONCE, for the selected LR only.
"""
import argparse
import datetime
import json
import os
import subprocess
import sys
import time

import numpy as np
import torch
from torch import nn
from sklearn.metrics import roc_auc_score, average_precision_score

ap = argparse.ArgumentParser()
ap.add_argument('--repo', required=True)
ap.add_argument('--cache', required=True, help='folder written by cache_embeddings.py')
ap.add_argument('--out', required=True, help='run folder (config, logs, metrics)')
ap.add_argument('--lrs', type=float, nargs='*', default=None, help='explicit LRs (otherwise sampled)')
ap.add_argument('--n_sweeps', type=int, default=10)
ap.add_argument('--sweep_seed', type=int, default=0, help='seed for sampling the LRs')
ap.add_argument('--seed', type=int, default=1002, help='training seed (init + shuffling)')
ap.add_argument('--epochs', type=int, default=100)
ap.add_argument('--patience', type=int, default=15)
ap.add_argument('--batch_size', type=int, default=16)
cli = ap.parse_args()

sys.path.insert(0, os.path.abspath(cli.repo))
from trainers.utils import get_model_performance  # noqa: E402  (repo's bootstrap CI)
import pandas as pd  # noqa: E402

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(cli.out, exist_ok=True)
log = open(os.path.join(cli.out, 'train.log'), 'a', encoding='utf-8')


def say(*a):
    s = ' '.join(str(x) for x in a)
    print(s, flush=True)
    log.write(s + '\n'); log.flush()


def load(split):
    return (torch.tensor(np.load(f'{cli.cache}/{split}_X.npy')),
            torch.tensor(np.load(f'{cli.cache}/{split}_y.npy')))


Xtr, ytr = load('train'); Xva, yva = load('val'); Xte, yte = load('test')
meta = json.load(open(f'{cli.cache}/meta.json'))
say(f'== {datetime.datetime.now():%Y-%m-%d %H:%M}  cache model={meta["model"]} modality={meta["modality"]} '
    f'notes={meta.get("notes_file")}  train/val/test = {len(Xtr)}/{len(Xva)}/{len(Xte)}  device={device}')


class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(768, 512)            # Text_encoder.fc_rr / fc_dn
        self.layernorm = nn.LayerNorm(512)       # MLPClassifier
        self.out = nn.Linear(512, ytr.shape[1])

    def forward(self, x):
        return self.out(self.layernorm(self.fc(x)))


def mean_auc(y, p):
    return float(np.mean([roc_auc_score(y[:, i], p[:, i]) for i in range(y.shape[1])]))


@torch.no_grad()
def predict(model, X):
    model.eval()
    return torch.cat([model(X[s:s + 1024].to(device)).cpu() for s in range(0, len(X), 1024)])


def run_one(lr):
    torch.manual_seed(cli.seed); np.random.seed(cli.seed)
    model = Head().to(device)
    opt = torch.optim.Adam(model.parameters(), lr, betas=(0.9, 0.9))
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=10, mode='min')
    lossf = nn.BCEWithLogitsLoss()
    g = torch.Generator().manual_seed(cli.seed)
    best, best_state, best_epoch, patience, hist = -1, None, -1, 0, []
    for epoch in range(cli.epochs):
        logits = predict(model, Xva)                                     # validate first (as repo)
        vloss = lossf(logits, yva).item()
        sched.step(vloss)
        auc = mean_auc(yva.numpy(), torch.sigmoid(logits).numpy())
        hist.append(round(auc, 4))
        if auc > best:
            best, best_epoch, patience = auc, epoch, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
        say(f'    lr={lr:.2e} epoch {epoch:3d}  val AUROC {auc:.4f}  val loss {vloss:.4f}'
            f'{"  *best*" if patience == 0 else ""}')
        model.train()                                                    # train epoch
        perm = torch.randperm(len(Xtr), generator=g)
        for s in range(0, len(perm) - cli.batch_size + 1, cli.batch_size):  # drop_last
            idx = perm[s:s + cli.batch_size]
            loss = lossf(model(Xtr[idx].to(device)), ytr[idx].to(device))
            opt.zero_grad(); loss.backward(); opt.step()
        if patience >= cli.patience:
            break
    return best, best_epoch, best_state, hist


if cli.lrs:
    lrs = cli.lrs
    dist = 'explicit'
else:
    rng = np.random.default_rng(cli.sweep_seed)
    lrs = list(10 ** rng.uniform(-5, -3, cli.n_sweeps))
    dist = f'log-uniform in [1e-5, 1e-3], sweep_seed={cli.sweep_seed}'
try:
    commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True,
                            cwd=cli.repo).stdout.strip()
except Exception:
    commit = 'unknown'
json.dump({'command': ' '.join(sys.argv), 'git_commit': commit, 'cache_meta': meta, 'lrs': lrs,
           'lr_distribution': dist, 'seed': cli.seed, 'epochs': cli.epochs, 'patience': cli.patience,
           'batch_size': cli.batch_size}, open(os.path.join(cli.out, 'config.json'), 'w'), indent=1)
say(f'LR sweep ({dist}): ' + ', '.join(f'{lr:.2e}' for lr in lrs))

results = []
for k, lr in enumerate(lrs):
    t0 = time.time()
    best, ep, state, hist = run_one(lr)
    results.append({'lr': lr, 'best_val_auroc': round(best, 4), 'best_epoch': ep, 'epochs_run': len(hist),
                    'val_auroc_by_epoch': hist, 'state': state})
    say(f'sweep {k + 1}/{len(lrs)}  lr={lr:.2e}  best val AUROC={best:.4f} at epoch {ep}  '
        f'({len(hist)} epochs, {time.time() - t0:.0f}s)')

chosen = max(results, key=lambda r: r['best_val_auroc'])             # selection on VALIDATION only
model = Head().to(device); model.load_state_dict(chosen['state'])
torch.save({'state_dict': chosen['state'], 'lr': chosen['lr'], 'cache_meta': meta},
           os.path.join(cli.out, 'best_head.pth'))

p = torch.sigmoid(predict(model, Xte)).numpy(); y = yte.numpy()       # TEST: once, chosen LR only
aucs, auprcs, rows = [], [], []
names = ['label_%d' % i for i in range(y.shape[1])]
try:
    from datasets.DataFusion import CLASSES
    names = CLASSES
except Exception:
    pass
for i in range(y.shape[1]):
    (ap_, ap_hi, ap_lo), (au, au_hi, au_lo) = get_model_performance(pd.DataFrame({'y_truth': y[:, i], 'y_pred': p[:, i]}))
    aucs.append(au); auprcs.append(ap_)
    rows.append({'label': names[i], 'auroc': round(au, 4), 'auroc_ci': [round(au_lo, 4), round(au_hi, 4)],
                 'auprc': round(ap_, 4), 'auprc_ci': [round(ap_lo, 4), round(ap_hi, 4)]})
summary = {'chosen_lr': chosen['lr'], 'chosen_val_auroc': chosen['best_val_auroc'],
           'test_mean_auroc': round(float(np.mean(aucs)), 4), 'test_mean_auprc': round(float(np.mean(auprcs)), 4),
           'per_label': rows,
           'sweeps': [{k: v for k, v in r.items() if k != 'state'} for r in results]}
json.dump(summary, open(os.path.join(cli.out, 'metrics.json'), 'w'), indent=1)
say(f'CHOSEN lr={chosen["lr"]:.2e} (val {chosen["best_val_auroc"]:.4f})  ->  '
    f'TEST mean AUROC {summary["test_mean_auroc"]:.4f}  mean AUPRC {summary["test_mean_auprc"]:.4f}')
pn = [r for r in rows if r['label'].startswith('Pneumonia')]
if pn:
    say(f'  pneumonia: AUROC {pn[0]["auroc"]} {pn[0]["auroc_ci"]}  AUPRC {pn[0]["auprc"]}')
say('metrics written to', os.path.abspath(os.path.join(cli.out, 'metrics.json')))

In [ ]:
import os, datetime

def fast(modality, limit=0, n_sweeps=10, epochs=100, lrs=None):
    # lrs=[x]: skip the 10-LR search and train only at learning rate x (used for extra seeds)
    mod = modality.upper()            # 'RR' or 'DN'
    m = mod.lower()
    tag = f'_limit{limit}' if limit else ''
    cache = f'{OUT}/cache/taskB_{m}_biobert{tag}'
    lr_tag = '_lr' + '_'.join(f'{x:.2e}' for x in lrs) if lrs else ''
    run_dir = f'{OUT}/runs/fast_taskB_unimodal_{m}_biobert_seed{SEED}{lr_tag}{tag}'
    os.makedirs(cache, exist_ok=True); os.makedirs(run_dir, exist_ok=True)
    md5 = open('/content/code_md5.txt').read().split()[0]
    # step A: BioBERT once per patient (skips splits and parts already saved, so it resumes after a disconnect)
    todo = [s for s in ['train', 'val', 'test'] if not os.path.exists(f'{cache}/{s}_X.npy')]
    if todo:
        cmd = (f'python /content/fast_text/cache_embeddings.py --repo /content/medpatch --modality {mod} '
               f'--bert_model_name {BERT} --ehr_data_dir /content/ehr --notes_data_dir /content/notes '
               f'--out "{cache}" --limit {limit} --splits {" ".join(todo)}')
        with open(f'{cache}/command.txt', 'a') as f:
            f.write(f'# {datetime.datetime.now():%Y-%m-%d %H:%M}  code md5: {md5}\n{cmd}\n\n')
        print('CACHE FOLDER:', cache)
        !cd /content/medpatch && {cmd} 2>&1 | grep --line-buffered -v "Warning\|warn(\|row_indexer\|pandas-docs" | tee -a "{cache}/cache.log"
    missing = [s for s in ['train', 'val', 'test'] if not os.path.exists(f'{cache}/{s}_X.npy')]
    assert not missing, f'caching did not finish for {missing}: run this cell again (it continues where it stopped)'
    # step B: train the small layer on the saved vectors (minutes per learning rate)
    cmd = (f'python /content/fast_text/train_head.py --repo /content/medpatch --cache "{cache}" --out "{run_dir}" '
           f'--seed {SEED} --n_sweeps {n_sweeps} --epochs {epochs}'
           + (' --lrs ' + ' '.join(repr(x) for x in lrs) if lrs else ''))
    with open(f'{run_dir}/command.txt', 'a') as f:
        f.write(f'# {datetime.datetime.now():%Y-%m-%d %H:%M}  code md5: {md5}\n{cmd}\n\n')
    print('RUN FOLDER:', run_dir)
    !cd /content/medpatch && {cmd} 2>&1 | grep --line-buffered -v "epoch .*val AUROC"
    print(f'\nper-epoch details: {run_dir}/train.log   all scores: {run_dir}/metrics.json')

## 8. FAST sanity test: 512 patients per split, 1 learning rate, 2 epochs (about 5–10 minutes)
Must finish without errors before the full fast runs. Its score means nothing.

In [ ]:
fast('RR', limit=512, n_sweeps=1, epochs=2)

## 9. FAST full run: Radiology Reports (target AUROC 0.776, AUPRC 0.474)
Step A (BioBERT once) takes a few hours on a T4; step B takes about 1 hour for the 10 learning rates.
If Colab disconnects: reconnect, run cells 1–7 again, then this cell. Step A continues from the last saved part.

In [ ]:
fast('RR')

## 10. FAST full run: Discharge Notes (target AUROC 0.856, AUPRC 0.601)
Discharge notes are longer than radiology reports, so step A takes longer. Same disconnect rule.

In [ ]:
fast('DN')

## 11. Extra seeds for DN (about 7 minutes per seed; needs step 10's cache)
Re-trains the small layer at the learning rate step 10 chose, with other seeds, then prints mean ± std
of the test scores (seed 1002 included). Step A is not repeated.

In [ ]:
import json, numpy as np
DN_LR = 0.00012225031390095305       # the CHOSEN lr from step 10 (exact value, from its config.json 'lrs')
EXTRA_SEEDS = [1, 2, 3, 4]           # plus 1002 from step 10 = 5 seeds
for s in EXTRA_SEEDS:
    SEED = s
    fast('DN', lrs=[DN_LR])
SEED = 1002
runs = [f'{OUT}/runs/fast_taskB_unimodal_dn_biobert_seed1002'] + \
       [f'{OUT}/runs/fast_taskB_unimodal_dn_biobert_seed{s}_lr{DN_LR:.2e}' for s in EXTRA_SEEDS]
res = [json.load(open(f'{r}/metrics.json')) for r in runs]
au = np.array([r['test_mean_auroc'] for r in res]); ap = np.array([r['test_mean_auprc'] for r in res])
print('per seed AUROC:', au, ' AUPRC:', ap)
print(f'DN 5 seeds: test AUROC {au.mean():.4f} ± {au.std(ddof=1):.4f}   AUPRC {ap.mean():.4f} ± {ap.std(ddof=1):.4f}')

## After each full run
Log in the team Sheet: Who=Farida · Date · What (unimodal RR / DN, **fast path**) · Task B phenotyping ·
File path (`PneumoVision_farida/runs/fast_taskB_unimodal_<rr|dn>_biobert_seed1002/best_head.pth`) ·
Score (test mean AUROC and AUPRC: last lines of the output, or `metrics.json`) ·
Notes (seed, chosen lr, "fast path: cached frozen BioBERT, dropout off", "restricted to repo split subjects").

---
## Optional: the original (slow) repo pipeline
About 2–2.5 hours per epoch on a T4 because it re-runs BioBERT on every patient every epoch. Not needed if you use
the fast path above.

In [ ]:
train('RR', ehr_dir='/content/ehr_small', run_name='SANITY_rr', epochs=1, resume=False)   # slow sanity test

In [ ]:
train('RR')

In [ ]:
train('DN')